# Evaluate the accuracy of an object-detection neural network 📈

This notebook is a simple demo on how to use the `eval` script to evaluate the accuracy of a model. Depending of its nature, it predicts: 
* a classification label ID (classifiers),
* one or multiple bounding box(es) to point to an object in an image (object-detection)
* a segmentation mask associated to an object in the target image source (segmentation)

In this notebook, the main idea is to evaluate the ability of a network to detect an object and its accuracy on a specific dataset with the following metrics:
* **Precision**: True Positives (TP) ratio with the sum of TP and False Positives (FP)
* **Recall**: True Positives (TP) ratio with the sum of TP and False Negatives (FN)
* **F1-score**: The harmonic mean of the precision and recall

Details can be found on scikit-learn documentation here: https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html or on wikipedia directly: https://en.wikipedia.org/wiki/Precision_and_recall

and finally the mean Average Precision (mAP), reported at specific Intersection over Union (IoU) thresholds. IoU measures the overlap between the predicted bounding box and the ground truth (actual) bounding box:
* **mAP50**: calculated using a fixed IoU threshold of 0.5.
* **mAP50-95**: calculated by averaging the mAP across multiple IoU thresholds, typically 10 points from 0.5 to 0.95.

as explained here: https://www.ultralytics.com/glossary/mean-average-precision-map

## GENERATE IR MODEL with KaNN
If you have not already, you will need a generated neural network using the `./generate` command.

NOTE: Particularly, we need the modules `input_preparator.py` and `output_preparator.py` that are generated with the network itself and inside the generated networks' folder. Without those modules, we will not be able to perform the metrics computation.

In [ ]:
%%bash
./generate ./networks/object-detection/yolov5s6-relu/onnx/network_f16.yaml -d yolov5s6 -f

## EVALUATE THE GENERATED MODEL
Perform the metrics computation with the next command. The arguments and syntax are:

`./eval <gen_dir> --metrics=<metrics> --dataset=<dataset> --device=<device>`

- `<gen_dir>`: the relative path to the generated network.
- `<metrics>`: the metric to analyse depending of the neural network to evaluate (top-1, mAP, mIoU)
- `<dataset>`: the name of the dataset to use (coco8, coco128 or coco).
- `<device>`: by default mppa, but can also be `cpu` for reference.

We will start gently with the smallest dataset: coco8 (8 images)

In [ ]:
%%bash
../eval yolov5s6 --metrics=mAP --dataset=coco8 --device=mppa

The interpretation of this result is as follows:
- P: precision TP/(TP + FP)
- R: recall TP/(TP + FN)
- F1 score: 2TP / (2TP + FP + FN)
- mAP50: mean average precision on a IoU threshold of 0.5.
- mAP50-95: mean average precision on a IoU threshold from 0.5 to 0.95 in steps of 0.05.

This evaluation can be compared with an inference on the CPU (using onnxruntime). Using the following command:

In [ ]:
%%bash
../eval yolov5s6 --metrics=mAP --dataset=coco8 --device=cpu

If the last metrics computation worked, we can move to the second biggest dataset: coco128.

In [ ]:
%%bash
../eval yolov5s6 --metrics=mAP --dataset=coco128

And finally, we can test the biggest dataset, named just coco, containing 5000 images.
> NOTE: this will stress your mppa for the inference step, then you RAM for loading the results
> from memory, and finally your CPU to compute the metrics and store their results. If too much RAM
> is used, your host device might malfunction or directly crash. There is a chunking mecanism that 
> should avoid this to happen, but we do not recommend using this program if you are already 
> consuming 40% of your RAM.

In [ ]:
%%bash
../eval yolov5s6 --metrics=map --dataset=coco